# 04_modeling_profile_wave2

Nouveau notebook de benchmark pour le **Projet 12**.

## Objectifs
1. Repartir d'une **nouvelle vague d'expérimentations MLflow** dédiée aux features de **profil de culture**.
2. Comparer trois branches :
   - **`current_reference`** : meilleure logique actuelle (enrichissement externe déjà testée dans le notebook précédent) ;
   - **`api_compatible`** : nouvelles features de profil **sans casser l'API actuelle** ;
   - **`api_breaking`** : nouvelles features de profil **avec variables supplémentaires** (`Soil_Type`, `Region`, `Weather_Condition`, `Days_to_Harvest`).
3. Faire un **mini benchmark multi-modèles** sur chaque branche :
   - Ridge linéaire
   - Ridge + interactions
   - RandomForest
   - CatBoost
4. Exporter les **deux gagnants** :
   - meilleur modèle **API-compatible**
   - meilleur modèle **API-breaking**

## Note méthodologique importante
Les features de profil internes sont ici calculées **globalement** à partir de `crop_aux`.
C'est acceptable pour une **vague exploratoire de benchmark**, mais cela reste légèrement optimiste.
Si cette piste est retenue, la prochaine étape sera d'industrialiser le calcul des profils **dans le train uniquement**.

In [1]:
from pathlib import Path
import sys
import json
import joblib
from copy import deepcopy

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from pathlib import Path
import project_paths as pp

PROCESSED_DIR = pp.PROCESSED_DIR

# Fallback si OUT_DIR n'existe pas dans project_paths.py
OUT_DIR = getattr(pp, "OUT_DIR", Path("outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

SEED = 42
SAMPLE_SIZE = 100_000

ARTIFACTS_DIR = ROOT / "artifacts" / "wave2_profile"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

WAVE2_OUT_DIR = OUT_DIR / "wave2_profile"
WAVE2_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("WAVE2_OUT_DIR:", WAVE2_OUT_DIR)

ROOT: C:\Users\thoma\Documents\Openclassroom\Projet-12
PROCESSED_DIR: C:\Users\thoma\Documents\Openclassroom\Projet-12\data\processed
ARTIFACTS_DIR: C:\Users\thoma\Documents\Openclassroom\Projet-12\artifacts\wave2_profile
WAVE2_OUT_DIR: outputs\wave2_profile


In [2]:
MLFLOW_DB_PATH = ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH.as_posix()}")
mlflow.set_experiment("projet-12-model-1-profile-wave-2")

print("MLflow version :", mlflow.__version__)
print("Tracking URI :", mlflow.get_tracking_uri())

MLflow version : 3.10.1
Tracking URI : sqlite:///C:/Users/thoma/Documents/Openclassroom/Projet-12/mlflow.db


## Chargement des données
Le notebook charge les CSV déjà produits par l'EDA.
Il reconstruit ensuite automatiquement les **features de profil internes** si elles ne sont pas déjà présentes.

In [3]:
crop_aux = pd.read_csv(PROCESSED_DIR / "crop_aux_clean.csv")
crop_aux_enriched = pd.read_csv(PROCESSED_DIR / "crop_aux_enriched.csv")

target_col = "Yield_tons_per_hectare"

print("crop_aux:", crop_aux.shape)
print("crop_aux_enriched:", crop_aux_enriched.shape)
display(crop_aux.head())
display(crop_aux_enriched.head())

crop_aux: (999769, 10)
crop_aux_enriched: (999769, 59)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,...,irrigation_profile_score,soil_matches_crop_modal_profile,region_matches_crop_modal_profile,weather_matches_crop_modal_profile,fertilizer_matches_crop_modal_profile,irrigation_matches_crop_modal_profile,mean_binary_profile_score,mean_categorical_profile_score,mean_context_profile_score,mean_modal_match_score
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,...,0.499622,1,0,0,0,0,0.499451,0.249712,0.349607,0.2
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,...,0.496504,1,0,0,1,0,0.498501,0.250467,0.349680,0.4
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,...,0.500345,1,1,0,1,1,0.500468,0.250227,0.350324,0.8
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573,...,0.500881,0,0,0,1,1,0.500499,0.249284,0.349770,0.4
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251,...,0.500906,0,0,1,1,1,0.501080,0.250832,0.350931,0.6


## Helpers généraux
On reprend la logique du notebook précédent et on l'étend pour gérer :
- les benchmarks multi-modèles,
- les CV MLflow,
- l'export des artefacts,
- la construction des features de profil.

In [4]:
def coerce_bool_like_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        1: True,
        0: False,
        True: True,
        False: False,
    }
    for col in columns:
        if col not in out.columns:
            continue
        if out[col].dtype == bool:
            continue
        out[col] = out[col].map(lambda x: mapping.get(str(x).strip().lower(), x) if pd.notna(x) else x)
        if out[col].dropna().isin([True, False]).all():
            out[col] = out[col].astype("boolean")
    return out


def split_feature_types(df: pd.DataFrame):
    bool_features = df.select_dtypes(include=["bool", "boolean"]).columns.tolist()
    numeric_features = df.select_dtypes(include=np.number).columns.tolist()
    numeric_features = numeric_features + [c for c in bool_features if c not in numeric_features]
    categorical_features = [c for c in df.columns if c not in numeric_features]
    return numeric_features, categorical_features


def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])


def make_pipeline(model, X: pd.DataFrame):
    numeric_features, categorical_features = split_feature_types(X)
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])


def make_interaction_pipeline(model, X: pd.DataFrame):
    numeric_features, categorical_features = split_feature_types(X)
    preprocessor = build_preprocessor(numeric_features, categorical_features)

    return Pipeline([
        ("preprocessor", preprocessor),
        ("interactions", PolynomialFeatures(
            degree=2,
            interaction_only=True,
            include_bias=False,
        )),
        ("model", model),
    ])


def evaluate_regression(y_true, y_pred, dataset_name: str, model_name: str) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    return {
        "dataset": dataset_name,
        "model": model_name,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


def holdout_metrics(pipe, X_train, X_test, y_train, y_test, dataset_name: str, model_name: str):
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    return evaluate_regression(y_test, preds, dataset_name, model_name), preds


def safe_log_model_params(model) -> dict:
    safe_params = {}
    if hasattr(model, "get_params"):
        for k, v in model.get_params().items():
            if isinstance(v, (str, int, float, bool, np.integer, np.floating)):
                safe_params[k] = v
    return safe_params


def cross_validate_pipeline(pipe, X, y, run_name: str, stage: str, extra_params: dict | None = None, notebook_name: str = "04_modeling_profile_wave2"):
    cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scoring = {
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    }

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    summary = {
        "rmse_mean": float(-scores["test_rmse"].mean()),
        "rmse_std": float((-scores["test_rmse"]).std()),
        "mae_mean": float(-scores["test_mae"].mean()),
        "mae_std": float((-scores["test_mae"]).std()),
        "r2_mean": float(scores["test_r2"].mean()),
        "r2_std": float(scores["test_r2"].std()),
    }

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "project": "Projet-12",
            "notebook": notebook_name,
            "stage": stage,
            "wave": "profile_wave2",
        })
        if extra_params:
            mlflow.log_params(extra_params)
        mlflow.log_metrics(summary)

    return summary


def export_model_with_meta(pipe, X_ref, y_ref, meta: dict, model_path: Path, meta_path: Path, run_name: str, notebook_name: str = "04_modeling_profile_wave2"):
    pipe.fit(X_ref, y_ref)
    signature = infer_signature(X_ref.head(100), pipe.predict(X_ref.head(100)))

    joblib.dump(pipe, model_path)

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "project": "Projet-12",
            "notebook": notebook_name,
            "stage": "export_model",
            "wave": "profile_wave2",
            "export_name": model_path.name,
        })
        params_to_log = {}
        for k, v in meta.items():
            if isinstance(v, (str, int, float, bool)):
                params_to_log[k] = v
        if params_to_log:
            mlflow.log_params(params_to_log)

        if "reference_metrics" in meta:
            for k, v in meta["reference_metrics"].items():
                if isinstance(v, (int, float)):
                    mlflow.log_metric(k, float(v))

        mlflow.sklearn.log_model(pipe, artifact_path="model", signature=signature)
        mlflow.log_artifact(str(meta_path))

## Construction / rafraîchissement des features de profil internes

Le notebook reconstruit ici les features de profil directement à partir de `crop_aux`.
Cela permet de rester autonome, même si `crop_aux_enriched.csv` n'a pas encore été régénéré depuis l'EDA.

In [5]:
crop_aux = coerce_bool_like_columns(crop_aux, ["Fertilizer_Used", "Irrigation_Used"])
crop_aux_enriched = coerce_bool_like_columns(crop_aux_enriched, ["Fertilizer_Used", "Irrigation_Used"])


def first_mode(series):
    modes = series.mode(dropna=True)
    if len(modes) == 0:
        return np.nan
    return modes.iloc[0]


def merge_crop_value_score(source_df, target_df, crop_col, value_col, score_col):
    counts = (
        source_df.groupby([crop_col, value_col], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
    totals = (
        source_df.groupby(crop_col, dropna=False)
        .size()
        .rename("n_crop")
        .reset_index()
    )

    score_df = counts.merge(totals, on=crop_col, how="left")
    score_df[score_col] = score_df["n"] / score_df["n_crop"]

    out = target_df.merge(
        score_df[[crop_col, value_col, score_col]],
        on=[crop_col, value_col],
        how="left",
    )
    return out


def add_internal_profile_features(source_df: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    out = target_df.copy()

    numeric_profile = (
        source_df.groupby("Crop", dropna=False)
        .agg(
            int_mean_rainfall_by_crop=("Rainfall_mm", "mean"),
            int_std_rainfall_by_crop=("Rainfall_mm", "std"),
            int_mean_temp_by_crop=("Temperature_Celsius", "mean"),
            int_std_temp_by_crop=("Temperature_Celsius", "std"),
            int_mean_days_to_harvest_by_crop=("Days_to_Harvest", "mean"),
            int_std_days_to_harvest_by_crop=("Days_to_Harvest", "std"),
        )
        .reset_index()
    )

    modal_profile = (
        source_df.groupby("Crop", dropna=False)
        .agg(
            soil_modal_crop_profile=("Soil_Type", first_mode),
            region_modal_crop_profile=("Region", first_mode),
            weather_modal_crop_profile=("Weather_Condition", first_mode),
            fertilizer_modal_crop_profile=("Fertilizer_Used", first_mode),
            irrigation_modal_crop_profile=("Irrigation_Used", first_mode),
        )
        .reset_index()
    )

    cols_to_drop = [
        "int_mean_rainfall_by_crop",
        "int_std_rainfall_by_crop",
        "int_mean_temp_by_crop",
        "int_std_temp_by_crop",
        "int_mean_days_to_harvest_by_crop",
        "int_std_days_to_harvest_by_crop",
        "soil_modal_crop_profile",
        "region_modal_crop_profile",
        "weather_modal_crop_profile",
        "fertilizer_modal_crop_profile",
        "irrigation_modal_crop_profile",
        "rainfall_gap_vs_int_crop_profile",
        "temp_gap_vs_int_crop_profile",
        "days_gap_vs_int_crop_profile",
        "abs_rainfall_gap_vs_int_crop_profile",
        "abs_temp_gap_vs_int_crop_profile",
        "abs_days_gap_vs_int_crop_profile",
        "z_abs_rainfall_gap_vs_int_crop_profile",
        "z_abs_temp_gap_vs_int_crop_profile",
        "z_abs_days_gap_vs_int_crop_profile",
        "numeric_profile_distance_z_2d",
        "numeric_profile_distance_z_3d",
        "soil_profile_score",
        "region_profile_score",
        "weather_profile_score",
        "fertilizer_profile_score",
        "irrigation_profile_score",
        "soil_matches_crop_modal_profile",
        "region_matches_crop_modal_profile",
        "weather_matches_crop_modal_profile",
        "fertilizer_matches_crop_modal_profile",
        "irrigation_matches_crop_modal_profile",
        "mean_binary_profile_score",
        "mean_categorical_profile_score",
        "mean_context_profile_score",
        "mean_modal_match_score",
    ]
    out = out.drop(columns=[c for c in cols_to_drop if c in out.columns], errors="ignore")

    out = out.merge(numeric_profile, on="Crop", how="left")
    out = out.merge(modal_profile, on="Crop", how="left")

    out["rainfall_gap_vs_int_crop_profile"] = out["Rainfall_mm"] - out["int_mean_rainfall_by_crop"]
    out["temp_gap_vs_int_crop_profile"] = out["Temperature_Celsius"] - out["int_mean_temp_by_crop"]
    out["days_gap_vs_int_crop_profile"] = out["Days_to_Harvest"] - out["int_mean_days_to_harvest_by_crop"]

    out["abs_rainfall_gap_vs_int_crop_profile"] = out["rainfall_gap_vs_int_crop_profile"].abs()
    out["abs_temp_gap_vs_int_crop_profile"] = out["temp_gap_vs_int_crop_profile"].abs()
    out["abs_days_gap_vs_int_crop_profile"] = out["days_gap_vs_int_crop_profile"].abs()

    safe_std_rain = out["int_std_rainfall_by_crop"].replace(0, np.nan)
    safe_std_temp = out["int_std_temp_by_crop"].replace(0, np.nan)
    safe_std_days = out["int_std_days_to_harvest_by_crop"].replace(0, np.nan)

    out["z_abs_rainfall_gap_vs_int_crop_profile"] = out["abs_rainfall_gap_vs_int_crop_profile"] / safe_std_rain
    out["z_abs_temp_gap_vs_int_crop_profile"] = out["abs_temp_gap_vs_int_crop_profile"] / safe_std_temp
    out["z_abs_days_gap_vs_int_crop_profile"] = out["abs_days_gap_vs_int_crop_profile"] / safe_std_days

    out["numeric_profile_distance_z_2d"] = np.sqrt(
        out["z_abs_rainfall_gap_vs_int_crop_profile"].fillna(0) ** 2
        + out["z_abs_temp_gap_vs_int_crop_profile"].fillna(0) ** 2
    )
    out["numeric_profile_distance_z_3d"] = np.sqrt(
        out["z_abs_rainfall_gap_vs_int_crop_profile"].fillna(0) ** 2
        + out["z_abs_temp_gap_vs_int_crop_profile"].fillna(0) ** 2
        + out["z_abs_days_gap_vs_int_crop_profile"].fillna(0) ** 2
    )

    out = merge_crop_value_score(source_df, out, "Crop", "Soil_Type", "soil_profile_score")
    out = merge_crop_value_score(source_df, out, "Crop", "Region", "region_profile_score")
    out = merge_crop_value_score(source_df, out, "Crop", "Weather_Condition", "weather_profile_score")
    out = merge_crop_value_score(source_df, out, "Crop", "Fertilizer_Used", "fertilizer_profile_score")
    out = merge_crop_value_score(source_df, out, "Crop", "Irrigation_Used", "irrigation_profile_score")

    for col in [
        "soil_profile_score",
        "region_profile_score",
        "weather_profile_score",
        "fertilizer_profile_score",
        "irrigation_profile_score",
    ]:
        out[col] = out[col].fillna(0.0)

    out["soil_matches_crop_modal_profile"] = (out["Soil_Type"] == out["soil_modal_crop_profile"]).astype(int)
    out["region_matches_crop_modal_profile"] = (out["Region"] == out["region_modal_crop_profile"]).astype(int)
    out["weather_matches_crop_modal_profile"] = (out["Weather_Condition"] == out["weather_modal_crop_profile"]).astype(int)
    out["fertilizer_matches_crop_modal_profile"] = (out["Fertilizer_Used"] == out["fertilizer_modal_crop_profile"]).astype(int)
    out["irrigation_matches_crop_modal_profile"] = (out["Irrigation_Used"] == out["irrigation_modal_crop_profile"]).astype(int)

    out["mean_binary_profile_score"] = out[["fertilizer_profile_score", "irrigation_profile_score"]].mean(axis=1)
    out["mean_categorical_profile_score"] = out[["soil_profile_score", "region_profile_score", "weather_profile_score"]].mean(axis=1)
    out["mean_context_profile_score"] = out[
        ["soil_profile_score", "region_profile_score", "weather_profile_score", "fertilizer_profile_score", "irrigation_profile_score"]
    ].mean(axis=1)
    out["mean_modal_match_score"] = out[
        [
            "soil_matches_crop_modal_profile",
            "region_matches_crop_modal_profile",
            "weather_matches_crop_modal_profile",
            "fertilizer_matches_crop_modal_profile",
            "irrigation_matches_crop_modal_profile",
        ]
    ].mean(axis=1)

    return out


crop_model_df = add_internal_profile_features(crop_aux, crop_aux_enriched)

print("crop_model_df:", crop_model_df.shape)
display(crop_model_df.head())
display(
    crop_model_df.groupby("Crop")[[
        "numeric_profile_distance_z_2d",
        "mean_binary_profile_score",
        "mean_categorical_profile_score",
        "mean_context_profile_score",
        "mean_modal_match_score",
    ]].mean().sort_index()
)

crop_model_df: (999769, 59)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,...,irrigation_profile_score,soil_matches_crop_modal_profile,region_matches_crop_modal_profile,weather_matches_crop_modal_profile,fertilizer_matches_crop_modal_profile,irrigation_matches_crop_modal_profile,mean_binary_profile_score,mean_categorical_profile_score,mean_context_profile_score,mean_modal_match_score
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,...,0.499622,1,0,0,0,0,0.499451,0.249712,0.349607,0.2
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,...,0.496504,1,0,0,1,0,0.498501,0.250467,0.349680,0.4
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,...,0.500345,1,1,0,1,1,0.500468,0.250227,0.350324,0.8
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573,...,0.500881,0,0,0,1,1,0.500499,0.249284,0.349770,0.4
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251,...,0.500906,0,0,1,1,1,0.501080,0.250832,0.350931,0.6


,numeric_profile_distance_z_2d,mean_binary_profile_score,mean_categorical_profile_score,mean_context_profile_score,mean_modal_match_score
Crop,,,,,
Barley,1.325280,0.500000,0.250002,0.350001,0.350711
Cotton,1.325503,0.500001,0.250005,0.350003,0.351033
Maize,1.325537,0.500002,0.250003,0.350003,0.350853
Rice,1.325201,0.500012,0.250004,0.350007,0.351515
Soybean,1.325206,0.500001,0.250009,0.350005,0.351303
Wheat,1.325145,0.500002,0.250008,0.350006,0.351443


## Définition des branches de benchmark

### `current_reference`
La logique la plus proche du meilleur notebook précédent.

### `api_compatible`
Ne demande pas de nouvelles variables à l'API :
- on reste sur `Crop`, `Rainfall_mm`, `Temperature_Celsius`, `Fertilizer_Used`, `Irrigation_Used`
- les features de profil sont dérivées côté serveur à partir de `Crop`

### `api_breaking`
Utilise en plus :
- `Soil_Type`
- `Region`
- `Weather_Condition`
- `Days_to_Harvest`
- et les scores de profil associés

In [6]:
current_reference_features = [
    "Crop",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
    "ext_mean_temp_by_crop",
    "ext_mean_rainfall_by_crop",
    "ext_mean_pesticides_by_crop",
    "ext_n_obs_by_crop",
    "rainfall_gap_vs_crop_profile",
    "temp_gap_vs_crop_profile",
]

api_compatible_features = [
    "Crop",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
    "int_mean_rainfall_by_crop",
    "int_mean_temp_by_crop",
    "rainfall_gap_vs_int_crop_profile",
    "temp_gap_vs_int_crop_profile",
    "abs_rainfall_gap_vs_int_crop_profile",
    "abs_temp_gap_vs_int_crop_profile",
    "z_abs_rainfall_gap_vs_int_crop_profile",
    "z_abs_temp_gap_vs_int_crop_profile",
    "fertilizer_profile_score",
    "irrigation_profile_score",
    "fertilizer_matches_crop_modal_profile",
    "irrigation_matches_crop_modal_profile",
    "mean_binary_profile_score",
    "numeric_profile_distance_z_2d",
]

api_breaking_features = [
    "Crop",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Days_to_Harvest",
    "Soil_Type",
    "Region",
    "Weather_Condition",
    "Fertilizer_Used",
    "Irrigation_Used",
    "int_mean_rainfall_by_crop",
    "int_std_rainfall_by_crop",
    "int_mean_temp_by_crop",
    "int_std_temp_by_crop",
    "int_mean_days_to_harvest_by_crop",
    "int_std_days_to_harvest_by_crop",
    "rainfall_gap_vs_int_crop_profile",
    "temp_gap_vs_int_crop_profile",
    "days_gap_vs_int_crop_profile",
    "abs_rainfall_gap_vs_int_crop_profile",
    "abs_temp_gap_vs_int_crop_profile",
    "abs_days_gap_vs_int_crop_profile",
    "z_abs_rainfall_gap_vs_int_crop_profile",
    "z_abs_temp_gap_vs_int_crop_profile",
    "z_abs_days_gap_vs_int_crop_profile",
    "numeric_profile_distance_z_2d",
    "numeric_profile_distance_z_3d",
    "soil_profile_score",
    "region_profile_score",
    "weather_profile_score",
    "fertilizer_profile_score",
    "irrigation_profile_score",
    "soil_matches_crop_modal_profile",
    "region_matches_crop_modal_profile",
    "weather_matches_crop_modal_profile",
    "fertilizer_matches_crop_modal_profile",
    "irrigation_matches_crop_modal_profile",
    "mean_binary_profile_score",
    "mean_categorical_profile_score",
    "mean_context_profile_score",
    "mean_modal_match_score",
]

feature_sets = {
    "current_reference": current_reference_features,
    "api_compatible": api_compatible_features,
    "api_breaking": api_breaking_features,
}

for name, cols in feature_sets.items():
    missing = [c for c in cols if c not in crop_model_df.columns]
    print(name, "| n_features =", len(cols), "| missing =", missing[:5], "..." if len(missing) > 5 else "")

current_reference | n_features = 11 | missing = [] 
api_compatible | n_features = 19 | missing = [] 
api_breaking | n_features = 40 | missing = [] 


## Définition des familles de modèles
On conserve un mini benchmark raisonnable :
- **Ridge linéaire**
- **Ridge + interactions**
- **RandomForest**
- **CatBoost**

L'objectif n'est pas ici de tout retuner finement, mais de voir si les nouvelles features sont mieux captées par une famille de modèles différente.

In [7]:
MODEL_SPECS = {
    "ridge": {
        "family": "linear",
        "model": Ridge(alpha=1.0),
        "extra_params": {
            "model_name": "Ridge",
            "alpha": 1.0,
        },
    },
    "ridge_interactions": {
        "family": "linear_interactions",
        "model": Ridge(alpha=1.0),
        "extra_params": {
            "model_name": "RidgeInteractions",
            "alpha": 1.0,
            "interaction_only": True,
            "degree": 2,
        },
    },
    "xgboost": {
        "family": "boosting",
        "model": XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=1,
        ),
        "extra_params": {
            "model_name": "XGBoost",
            "n_estimators": 300,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
        },
    },
    "catboost": {
        "family": "boosting",
        "model": CatBoostRegressor(
            iterations=300,
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            verbose=0,
            random_seed=SEED,
        ),
        "extra_params": {
            "model_name": "CatBoost",
            "iterations": 300,
            "depth": 6,
            "learning_rate": 0.05,
        },
    },
}

def build_pipe_from_spec(spec, X):
    family = spec["family"]
    model = spec["model"]

    if family == "linear":
        return make_pipeline(model, X)

    if family == "linear_interactions":
        return make_interaction_pipeline(model, X)

    if family == "boosting":
        return make_pipeline(model, X)

    raise ValueError(f"Unknown family: {family}")

## Échantillon de travail pour le mini benchmark
On garde un échantillon pour un premier tri rapide.

In [8]:
sample_df = crop_model_df.sample(
    n=min(SAMPLE_SIZE, len(crop_model_df)),
    random_state=SEED,
).copy()

y_sample = sample_df[target_col].copy()
print("sample_df:", sample_df.shape)

sample_df: (100000, 59)


## Benchmark holdout sur échantillon
Chaque branche est évaluée avec les 4 familles de modèles.
Tous les runs sont loggés dans MLflow avec le tag `wave=profile_wave2`.

In [9]:
def benchmark_feature_set_on_sample(
    df: pd.DataFrame,
    target_col: str,
    feature_set_name: str,
    feature_columns: list[str],
    model_specs: dict,
    notebook_name: str = "04_modeling_profile_wave2",
):
    X = df[feature_columns].copy()
    y = df[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
    )

    rows = []

    for model_key, spec in model_specs.items():
        pipe = build_pipe_from_spec(spec, X_train)
        result, _ = holdout_metrics(
            pipe,
            X_train,
            X_test,
            y_train,
            y_test,
            dataset_name=feature_set_name,
            model_name=model_key,
        )

        row = {
            "feature_set": feature_set_name,
            "model_key": model_key,
            "stage": "sample_holdout",
            "n_features": len(feature_columns),
            **result,
        }
        rows.append(row)

        with mlflow.start_run(run_name=f"wave2__sample__{feature_set_name}__{model_key}"):
            mlflow.set_tags({
                "project": "Projet-12",
                "notebook": notebook_name,
                "stage": "sample_holdout",
                "wave": "profile_wave2",
                "feature_set": feature_set_name,
                "model_key": model_key,
                "api_mode": feature_set_name,
            })
            mlflow.log_param("feature_set", feature_set_name)
            mlflow.log_param("model_key", model_key)
            mlflow.log_param("n_features", len(feature_columns))
            mlflow.log_param("feature_names", ",".join(feature_columns))
            mlflow.log_param("n_train_rows", len(X_train))
            mlflow.log_param("n_test_rows", len(X_test))
            mlflow.log_params(spec["extra_params"])
            mlflow.log_metrics({
                "rmse": float(result["rmse"]),
                "mae": float(result["mae"]),
                "r2": float(result["r2"]),
            })

    return pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)


sample_benchmark_frames = []
for feature_set_name, feature_columns in feature_sets.items():
    result_df = benchmark_feature_set_on_sample(
        df=sample_df,
        target_col=target_col,
        feature_set_name=feature_set_name,
        feature_columns=feature_columns,
        model_specs=MODEL_SPECS,
    )
    sample_benchmark_frames.append(result_df)

sample_benchmark_results = pd.concat(sample_benchmark_frames, ignore_index=True)
display(sample_benchmark_results.sort_values(["feature_set", "rmse"]).reset_index(drop=True))

sample_benchmark_results.to_csv(WAVE2_OUT_DIR / "sample_benchmark_results.csv", index=False)

,feature_set,model_key,stage,n_features,dataset,model,rmse,mae,r2
0,api_breaking,ridge,sample_holdout,40,api_breaking,ridge,0.501128,0.399617,0.912139
1,api_breaking,catboost,sample_holdout,40,api_breaking,catboost,0.502188,0.400457,0.911767
2,api_breaking,ridge_interactions,sample_holdout,40,api_breaking,ridge_interactions,0.503045,0.401093,0.911465
3,api_breaking,xgboost,sample_holdout,40,api_breaking,xgboost,0.505017,0.402442,0.910770
4,api_compatible,ridge,sample_holdout,19,api_compatible,ridge,0.501161,0.399615,0.912127
5,api_compatible,ridge_interactions,sample_holdout,19,api_compatible,ridge_interactions,0.501363,0.399844,0.912057
6,api_compatible,catboost,sample_holdout,19,api_compatible,catboost,0.501739,0.400072,0.911925
7,api_compatible,xgboost,sample_holdout,19,api_compatible,xgboost,0.503885,0.401484,0.911170
8,current_reference,ridge,sample_holdout,11,current_reference,ridge,0.501083,0.399569,0.912155
9,current_reference,ridge_interactions,sample_holdout,11,current_reference,ridge_interactions,0.501145,0.399653,0.912133


## Benchmark CV complet
On relance ensuite une CV 5 folds sur **toutes** les combinaisons retenues dans cette vague.

In [10]:
full_df = crop_model_df.copy()
y_full = full_df[target_col].copy()


def benchmark_feature_set_cv(
    df: pd.DataFrame,
    target_col: str,
    feature_set_name: str,
    feature_columns: list[str],
    model_specs: dict,
    notebook_name: str = "04_modeling_profile_wave2",
):
    X = df[feature_columns].copy()
    y = df[target_col].copy()

    rows = []
    fitted_pipes = {}

    for model_key, spec in model_specs.items():
        pipe = build_pipe_from_spec(spec, X)

        summary = cross_validate_pipeline(
            pipe=pipe,
            X=X,
            y=y,
            run_name=f"wave2__cv__{feature_set_name}__{model_key}",
            stage="full_cv",
            extra_params={
                "feature_set": feature_set_name,
                "model_key": model_key,
                "n_features": len(feature_columns),
                **spec["extra_params"],
            },
            notebook_name=notebook_name,
        )

        row = {
            "feature_set": feature_set_name,
            "model_key": model_key,
            "stage": "full_cv",
            "n_features": len(feature_columns),
            **summary,
        }
        rows.append(row)
        fitted_pipes[model_key] = pipe

    return pd.DataFrame(rows).sort_values("rmse_mean").reset_index(drop=True), fitted_pipes


cv_benchmark_frames = []
feature_set_pipes = {}

for feature_set_name, feature_columns in feature_sets.items():
    result_df, pipe_dict = benchmark_feature_set_cv(
        df=full_df,
        target_col=target_col,
        feature_set_name=feature_set_name,
        feature_columns=feature_columns,
        model_specs=MODEL_SPECS,
    )
    cv_benchmark_frames.append(result_df)
    feature_set_pipes[feature_set_name] = pipe_dict

cv_benchmark_results = pd.concat(cv_benchmark_frames, ignore_index=True)
display(cv_benchmark_results.sort_values(["feature_set", "rmse_mean"]).reset_index(drop=True))

cv_benchmark_results.to_csv(WAVE2_OUT_DIR / "cv_benchmark_results.csv", index=False)

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\thoma\Documents\Openclassroom\Projet-12\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\thoma\Documents\Openclassroom\Projet-12\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\thoma\Documents\Openclassroom\Projet-12\.venv\Lib\site-packages\sklearn\pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "c:\Users\thoma\Documents\Openclassroom\Pr

## Lecture rapide des meilleurs candidats

In [ ]:
best_by_feature_set = (
    cv_benchmark_results.sort_values(["feature_set", "rmse_mean"])
    .groupby("feature_set", as_index=False)
    .first()
)

display(best_by_feature_set)

best_current_reference = best_by_feature_set[best_by_feature_set["feature_set"] == "current_reference"].iloc[0].to_dict()
best_api_compatible = best_by_feature_set[best_by_feature_set["feature_set"] == "api_compatible"].iloc[0].to_dict()
best_api_breaking = best_by_feature_set[best_by_feature_set["feature_set"] == "api_breaking"].iloc[0].to_dict()

print("Best current_reference :", best_current_reference["model_key"], "| RMSE =", best_current_reference["rmse_mean"])
print("Best api_compatible   :", best_api_compatible["model_key"], "| RMSE =", best_api_compatible["rmse_mean"])
print("Best api_breaking     :", best_api_breaking["model_key"], "| RMSE =", best_api_breaking["rmse_mean"])

## Fit des gagnants API-compatible et API-breaking
Ces deux modèles sont les vrais candidats exportables de cette vague.

In [ ]:
def get_best_pipe_and_data(feature_set_name: str, best_row: dict):
    feature_columns = feature_sets[feature_set_name]
    X = full_df[feature_columns].copy()
    y = full_df[target_col].copy()
    pipe = build_pipe_from_spec(MODEL_SPECS[best_row["model_key"]], X)
    pipe.fit(X, y)
    return pipe, X, y, feature_columns


api_compatible_pipe, X_api_compatible_full, y_api_compatible_full, api_compatible_feature_names = get_best_pipe_and_data(
    "api_compatible",
    best_api_compatible,
)

api_breaking_pipe, X_api_breaking_full, y_api_breaking_full, api_breaking_feature_names = get_best_pipe_and_data(
    "api_breaking",
    best_api_breaking,
)

print("API-compatible winner:", best_api_compatible["model_key"])
print("API-breaking winner  :", best_api_breaking["model_key"])

## Snapshots de ranking par culture

Pour comparer visuellement les deux gagnants, on construit des scénarios types :
- **API-compatible** : on fixe `Rainfall_mm`, `Temperature_Celsius`, `Fertilizer_Used`, `Irrigation_Used`
- **API-breaking** : on complète en plus avec les **valeurs modales / moyennes du profil** si l'utilisateur ne renseigne pas explicitement les variables supplémentaires

In [ ]:
internal_numeric_profile = (
    crop_aux.groupby("Crop", dropna=False)
    .agg(
        int_mean_rainfall_by_crop=("Rainfall_mm", "mean"),
        int_std_rainfall_by_crop=("Rainfall_mm", "std"),
        int_mean_temp_by_crop=("Temperature_Celsius", "mean"),
        int_std_temp_by_crop=("Temperature_Celsius", "std"),
        int_mean_days_to_harvest_by_crop=("Days_to_Harvest", "mean"),
        int_std_days_to_harvest_by_crop=("Days_to_Harvest", "std"),
    )
    .reset_index()
)

internal_modal_profile = (
    crop_aux.groupby("Crop", dropna=False)
    .agg(
        soil_modal_crop_profile=("Soil_Type", first_mode),
        region_modal_crop_profile=("Region", first_mode),
        weather_modal_crop_profile=("Weather_Condition", first_mode),
        fertilizer_modal_crop_profile=("Fertilizer_Used", first_mode),
        irrigation_modal_crop_profile=("Irrigation_Used", first_mode),
    )
    .reset_index()
)

external_profile_cols = [
    "Crop",
    "ext_mean_temp_by_crop",
    "ext_mean_rainfall_by_crop",
    "ext_mean_pesticides_by_crop",
    "ext_n_obs_by_crop",
]
external_profiles = (
    crop_model_df[external_profile_cols]
    .drop_duplicates(subset=["Crop"])
    .copy()
)


def build_crop_scenario_frame(
    crop_list,
    rainfall_mm=1200,
    temperature_celsius=24,
    fertilizer_used=True,
    irrigation_used=True,
    days_to_harvest=None,
    soil_type=None,
    region=None,
    weather_condition=None,
):
    scenario = pd.DataFrame({"Crop": crop_list})
    scenario["Rainfall_mm"] = rainfall_mm
    scenario["Temperature_Celsius"] = temperature_celsius
    scenario["Fertilizer_Used"] = fertilizer_used
    scenario["Irrigation_Used"] = irrigation_used

    scenario = scenario.merge(internal_numeric_profile, on="Crop", how="left")
    scenario = scenario.merge(internal_modal_profile, on="Crop", how="left")
    scenario = scenario.merge(external_profiles, on="Crop", how="left")

    scenario["Days_to_Harvest"] = (
        scenario["int_mean_days_to_harvest_by_crop"] if days_to_harvest is None else days_to_harvest
    )
    scenario["Soil_Type"] = (
        scenario["soil_modal_crop_profile"] if soil_type is None else soil_type
    )
    scenario["Region"] = (
        scenario["region_modal_crop_profile"] if region is None else region
    )
    scenario["Weather_Condition"] = (
        scenario["weather_modal_crop_profile"] if weather_condition is None else weather_condition
    )

    scenario["rainfall_gap_vs_crop_profile"] = scenario["Rainfall_mm"] - scenario["ext_mean_rainfall_by_crop"]
    scenario["temp_gap_vs_crop_profile"] = scenario["Temperature_Celsius"] - scenario["ext_mean_temp_by_crop"]

    scenario = add_internal_profile_features(crop_aux, scenario)

    return scenario


def crop_ranking_snapshot(pipe, feature_columns, dataset_name: str, **scenario_kwargs):
    crops = sorted(crop_model_df["Crop"].dropna().astype(str).unique())
    scenario = build_crop_scenario_frame(crops, **scenario_kwargs)

    missing_cols = [c for c in feature_columns if c not in scenario.columns]
    if missing_cols:
        raise KeyError(f"Colonnes manquantes dans le scénario {dataset_name}: {missing_cols}")

    scenario_model = scenario[feature_columns].copy()
    preds = pipe.predict(scenario_model)

    out = pd.DataFrame({
        "Crop": crops,
        "predicted_yield_t_ha": preds,
    }).sort_values("predicted_yield_t_ha", ascending=False).reset_index(drop=True)

    out["dataset_name"] = dataset_name
    out["gap_vs_top1"] = out["predicted_yield_t_ha"].iloc[0] - out["predicted_yield_t_ha"]
    return out

In [ ]:
ranking_api_compatible = crop_ranking_snapshot(
    api_compatible_pipe,
    api_compatible_feature_names,
    "api_compatible_winner",
)

ranking_api_breaking = crop_ranking_snapshot(
    api_breaking_pipe,
    api_breaking_feature_names,
    "api_breaking_winner",
)

display(ranking_api_compatible)
display(ranking_api_breaking)

ranking_api_compatible.to_csv(WAVE2_OUT_DIR / "ranking_api_compatible.csv", index=False)
ranking_api_breaking.to_csv(WAVE2_OUT_DIR / "ranking_api_breaking.csv", index=False)

## Export des gagnants
On exporte **les deux modèles** pour pouvoir comparer ensuite :
- un candidat **sans casser l'API**
- un candidat **avec API étendue**

In [ ]:
api_compatible_model_path = ARTIFACTS_DIR / "model_1_crop_api_compatible_wave2.joblib"
api_compatible_meta_path = ARTIFACTS_DIR / "model_1_crop_api_compatible_wave2_meta.json"

api_breaking_model_path = ARTIFACTS_DIR / "model_1_crop_api_breaking_wave2.joblib"
api_breaking_meta_path = ARTIFACTS_DIR / "model_1_crop_api_breaking_wave2_meta.json"

api_compatible_meta = {
    "model_name": MODEL_SPECS[best_api_compatible["model_key"]]["extra_params"]["model_name"],
    "variant_name": "api_compatible_wave2",
    "model_key": best_api_compatible["model_key"],
    "target_name": target_col,
    "target_unit": "t/ha",
    "feature_names": api_compatible_feature_names,
    "training_dataset": "crop_model_df",
    "api_mode": "compatible",
    "api_requires_new_fields": False,
    "cv_summary": {
        k: float(v) for k, v in best_api_compatible.items()
        if k in ["rmse_mean", "rmse_std", "mae_mean", "mae_std", "r2_mean", "r2_std"]
    },
    "reference_metrics": {
        k: float(v) for k, v in best_api_compatible.items()
        if k in ["rmse_mean", "rmse_std", "mae_mean", "mae_std", "r2_mean", "r2_std"]
    },
    "error_margin_t_ha": float(best_api_compatible["rmse_mean"]),
    "supported_crops": sorted(crop_model_df["Crop"].dropna().astype(str).unique().tolist()),
    "crop_support_counts": (
        crop_model_df["Crop"]
        .dropna()
        .astype(str)
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "selection_rationale": (
        "Gagnant wave2 parmi les variantes API-compatibles basées sur les features de profil internes."
    ),
    "preprocessing_note": "Prétraitement pipeline auto + éventuels profils dérivés côté serveur.",
}

api_breaking_meta = {
    "model_name": MODEL_SPECS[best_api_breaking["model_key"]]["extra_params"]["model_name"],
    "variant_name": "api_breaking_wave2",
    "model_key": best_api_breaking["model_key"],
    "target_name": target_col,
    "target_unit": "t/ha",
    "feature_names": api_breaking_feature_names,
    "training_dataset": "crop_model_df",
    "api_mode": "breaking",
    "api_requires_new_fields": True,
    "required_api_fields": ["crop", "rainfall_mm", "temperature_celsius", "days_to_harvest", "soil_type", "region", "weather_condition", "fertilizer_used", "irrigation_used"],
    "cv_summary": {
        k: float(v) for k, v in best_api_breaking.items()
        if k in ["rmse_mean", "rmse_std", "mae_mean", "mae_std", "r2_mean", "r2_std"]
    },
    "reference_metrics": {
        k: float(v) for k, v in best_api_breaking.items()
        if k in ["rmse_mean", "rmse_std", "mae_mean", "mae_std", "r2_mean", "r2_std"]
    },
    "error_margin_t_ha": float(best_api_breaking["rmse_mean"]),
    "supported_crops": sorted(crop_model_df["Crop"].dropna().astype(str).unique().tolist()),
    "crop_support_counts": (
        crop_model_df["Crop"]
        .dropna()
        .astype(str)
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "selection_rationale": (
        "Gagnant wave2 parmi les variantes API-breaking avec variables contextuelles complètes et scores de profil."
    ),
    "preprocessing_note": "Prétraitement pipeline auto + variables contextuelles complètes + scores de profil internes.",
}

export_model_with_meta(
    pipe=api_compatible_pipe,
    X_ref=X_api_compatible_full,
    y_ref=y_api_compatible_full,
    meta=api_compatible_meta,
    model_path=api_compatible_model_path,
    meta_path=api_compatible_meta_path,
    run_name="export_model_1_crop_api_compatible_wave2",
)

export_model_with_meta(
    pipe=api_breaking_pipe,
    X_ref=X_api_breaking_full,
    y_ref=y_api_breaking_full,
    meta=api_breaking_meta,
    model_path=api_breaking_model_path,
    meta_path=api_breaking_meta_path,
    run_name="export_model_1_crop_api_breaking_wave2",
)

print("Exports réalisés :")
print("-", api_compatible_model_path)
print("-", api_breaking_model_path)

## Résumé opérationnel à remplir après exécution

À la fin de l'exécution, complète si besoin :

1. **Le meilleur modèle global** de la wave 2 est : `...`
2. **Le meilleur modèle API-compatible** est : `...`
3. **Le meilleur modèle API-breaking** est : `...`
4. Décision produit :
   - [ ] on garde l'API actuelle
   - [ ] on fait évoluer l'API
5. Prochaine étape :
   - [ ] brancher le gagnant dans l'API
   - [ ] durcir le calcul des profils sans fuite (train only / fold only)
   - [ ] ajouter un benchmark de recommandation top-k par culture